# 💬 Messages: The Unit of Context

## Learning Objectives
In this notebook, you will learn:
1. **Text vs. message prompts** - when a bare string is enough and when you need message objects
2. **The four message types** - `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`
3. **Message metadata** - `name`, `id`, and what they are actually for
4. **`usage_metadata`** - reading token counts off a response to track cost
5. **Hand-building history** - inserting AI and tool messages as if the model had produced them
6. **Messages in an agent run** - what `create_agent` returns, and why one AI message looks empty

## Prerequisites
- Chat models — `01_model_integration.ipynb`; the tool-call loop — `02_tool_calling_manual_loop.ipynb` (module 02)
- Dependencies installed with `uv sync` at the repository root (LangChain **1.x**)
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the key, confirm LangChain 1.x (the `langchain.messages` and `langchain.agents` imports below do not exist on 0.x), and create the model.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

import langchain

# On 0.x, `langchain.agents.create_agent` does not exist - fail here, not three cells later.
assert langchain.__version__.startswith("1."), "This notebook requires LangChain 1.x"
print(f"✅ LangChain version: {langchain.__version__}")

print("✅ Environment ready")

✅ LangChain version: 1.4.3
✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: The model every message is sent to
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
## 💡 Part 1: What Is a Message?

Messages are the fundamental unit of context for models in LangChain. They represent both the
input and the output of a model, carrying the content *and* the metadata needed to represent
the state of a conversation.

Every message object contains:

- **Role** — identifies the message type (system, user, assistant, tool)
- **Content** — the actual payload: text, images, audio, documents
- **Metadata** — optional fields such as response info, message IDs, and token usage

### Key Insight:
LangChain provides one standard message type that works across **all** providers. OpenAI,
Anthropic, and Google each wire messages differently on the API; you write against the
LangChain types and the integration package translates. Swapping providers does not change
your message-handling code.

---
## 📝 Part 2: Text Prompts

The simplest input is a plain string. LangChain wraps it into a single `HumanMessage` for you.

**Use text prompts when:**
- You have a single, standalone request
- You do not need conversation history
- You want minimal code complexity

In [3]:
# ============================================================================
# TEXT PROMPT: A bare string is wrapped into a HumanMessage automatically
# ============================================================================
response = llm.invoke("In two sentences: what is artificial intelligence?")

print(f"📋 type: {type(response).__name__}")
print("🤖", response.content)

📋 type: AIMessage
🤖 Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems, enabling them to perform tasks such as learning, reasoning, and problem-solving. It encompasses various technologies, including machine learning, natural language processing, and robotics, aimed at creating systems that can operate autonomously or assist humans in decision-making.


---
## 🧱 Part 3: Message Prompts — the Four Types

For anything beyond a one-shot question you pass a **list of message objects**. There are four
types, and each has a distinct job:

| Type | Role | Purpose |
|------|------|---------|
| `SystemMessage` | system | Tells the model how to behave; sets tone, role, and guidelines |
| `HumanMessage` | user | User input — text, images, audio, files, any multimodal content |
| `AIMessage` | assistant | Model output — text content, tool calls, and provider metadata |
| `ToolMessage` | tool | The result of a single tool execution, fed back to the model |

> **Note**: the list *is* the conversation state. There is no hidden memory — whatever you pass
> is exactly what the model sees, which is why message construction is the core skill here.

### 3.1 ⚙️ System Message

A `SystemMessage` is an initial set of instructions that primes the model's behaviour. Use it
to set the tone, define the model's role, and establish guidelines for responses.

In [4]:
# ============================================================================
# SYSTEM MESSAGE: Priming the model's role
# ============================================================================
from langchain.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence"),
]

response = llm.invoke(messages)
print("🤖", response.content)

🤖 In circuits woven, thoughts ignite,  
A spark of code, a dance of light,  
From silicon dreams, a mind takes flight,  
In realms of data, both day and night.  

With algorithms sharp, it learns and grows,  
In patterns hidden, the wisdom flows,  
From language vast to artful prose,  
A mirror reflecting what humanity knows.  

Yet in this creation, a question stirs,  
What is the heart behind the wires?  
Can it feel the pulse of human fears,  
Or grasp the warmth of laughter's cheers?  

In endless queries, it seeks to find,  
The essence of love, the threads that bind,  
But in its logic, so clear and refined,  
Lies a void where the soul is confined.  

Oh, child of progress, bright and bold,  
You weave our stories, both new and old,  
Yet remember, dear friend, as you unfold,  
The heart of creation is not bought or sold.  

So let us dance in this digital age,  
With wisdom and wonder, turn the page,  
For in our union, we can engage,  
A future where both can share the stage. 

In [5]:
# ============================================================================
# SYSTEM MESSAGE: Same question, a different role -> a different answer
# ============================================================================
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?"),
]

response = llm.invoke(messages)
print("🤖", response.content[:600], "...")

🤖 Creating a REST API involves several steps, including planning your API, choosing a technology stack, implementing the API, and testing it. Below is a general guide to help you create a REST API.

### Step 1: Plan Your API

1. **Define the Purpose**: Determine what resources your API will expose and what operations will be available (e.g., CRUD operations).
2. **Identify Resources**: List the resources (e.g., users, products) and their relationships.
3. **Design Endpoints**: Define the endpoints for your API. For example:
   - `GET /users` - Retrieve a list of users
   - `POST /users` - Create ...


#### 🎨 Making the System Message Do More Work

The more specific the system message, the less steering each individual request needs. Compare
the output below with the terse version above — same user question, but the instructions now
pin down expertise level, format, and verbosity.

In [6]:
# ============================================================================
# SYSTEM MESSAGE: Detailed instructions shape format, not just tone
# ============================================================================
system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?"),
]

response = llm.invoke(messages)
print("🤖", response.content[:800], "...")

🤖 Creating a REST API in Python can be efficiently done using a web framework like Flask or FastAPI. Below, I'll provide a step-by-step guide using both frameworks, starting with Flask.

### Using Flask

1. **Install Flask**:
   First, you need to install Flask. You can do this using pip:
   ```bash
   pip install Flask
   ```

2. **Create a Basic Flask Application**:
   Here’s a simple example of a REST API that manages a list of items.

   ```python
   from flask import Flask, jsonify, request

   app = Flask(__name__)

   # Sample data
   items = [
       {'id': 1, 'name': 'Item 1'},
       {'id': 2, 'name': 'Item 2'}
   ]

   @app.route('/items', methods=['GET'])
   def get_items():
       return jsonify(items)

   @app.route('/items/<int:item_id>', methods=['GET'])
   def get_item(item_ ...


### 3.2 👤 Human Message and Metadata

A `HumanMessage` represents user input. Beyond `content`, two optional fields matter in real
applications:

- **`name`** — identifies *which* user spoke, in a multi-user or multi-persona conversation
- **`id`** — a stable identifier you choose, so a message can be traced through logs and
  observability tooling

> **Note**: these are metadata for *your* systems. Most providers ignore `id` entirely, and
> support for `name` varies — never rely on it to carry instructions to the model.

In [7]:
# ============================================================================
# MESSAGE METADATA: Optional name and id fields
# ============================================================================
human_msg = HumanMessage(
    content="Hello!",
    name="alice",   # Optional: identify different users in one conversation
    id="msg_123",   # Optional: unique identifier for tracing
)

print(f"📋 content: {human_msg.content}")
print(f"📋 name:    {human_msg.name}")
print(f"📋 id:      {human_msg.id}")

📋 content: Hello!
📋 name:    alice
📋 id:      msg_123


In [8]:
# ============================================================================
# INVOCATION: Metadata rides along; the model answers the content
# ============================================================================
response = llm.invoke([human_msg])
print("🤖", response.content)

🤖 Hello, Alice! How can I assist you today?


### 3.3 🤖 AI Message

An `AIMessage` is the output of a model invocation. It can carry multimodal data, tool calls,
and provider-specific metadata.

You can also **construct one yourself** and insert it into the history — which is how you
replay a stored conversation, or seed the model with a response you want it to treat as its
own. The model cannot tell the difference.

In [9]:
# ============================================================================
# AI MESSAGE: Hand-built and inserted into the conversation history
# ============================================================================
ai_msg = AIMessage("I'd be happy to help you with that question!")

messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,                              # inserted as if the model had said it
    HumanMessage("Great! What's 2+2?"),
]

response = llm.invoke(messages)
print("🤖", response.content)

🤖 2 + 2 equals 4.


#### 📊 Token Usage

Every response carries `usage_metadata` with the token counts for that call. This is the
number to log if you care about cost — input and output tokens are billed at different rates,
so the split matters.

In [10]:
# ============================================================================
# TOKEN USAGE: input / output / total for the call above
# ============================================================================
usage = response.usage_metadata

print(f"📊 input tokens:  {usage['input_tokens']}")
print(f"📊 output tokens: {usage['output_tokens']}")
print(f"📊 total tokens:  {usage['total_tokens']}")

📊 input tokens:  47
📊 output tokens: 8
📊 total tokens:  55


### 3.4 🔧 Tool Message

For models that support tool calling, an `AIMessage` can contain **tool calls**. A
`ToolMessage` then passes the result of a single tool execution back to the model.

### Key Concepts:
- **`tool_call_id` must match**: the `ToolMessage`'s `tool_call_id` has to equal the `id` of
  the `AIMessage` tool call it answers. That pairing is how the model matches results to
  requests when several tools were called at once — mismatch it and the provider errors out.
- **`content=[]` on the AI message**: a pure tool-call turn has no text, which is exactly the
  "empty AI message" you will meet again in Part 4.

Below we hand-build the whole exchange rather than running a real tool, so the structure is
visible in one cell.

In [11]:
# ============================================================================
# TOOL MESSAGE: Hand-building an AIMessage -> ToolMessage exchange
# ============================================================================
from langchain.messages import ToolMessage

# The model's tool-call turn: no text content, only a request.
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123",
    }],
)

# What executing that tool produced.
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123",  # ⚠️ must match the call ID above
)

messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,     # the model's tool call
    tool_message,   # the tool's result
]

response = llm.invoke(messages)  # the model now answers from the result
print("🤖", response.content)

🤖 The weather in San Francisco is sunny with a temperature of 72°F.


In [12]:
# ============================================================================
# INSPECT: The assembled conversation, one line per message
# ============================================================================
for i, m in enumerate(messages + [response]):
    extra = f"  tool_calls={m.tool_calls}" if getattr(m, "tool_calls", None) else ""
    extra += f"  tool_call_id={m.tool_call_id}" if isinstance(m, ToolMessage) else ""
    print(f"[{i}] {type(m).__name__:<13} content={m.content!r}{extra}")

[0] HumanMessage  content="What's the weather in San Francisco?"
[1] AIMessage     content=[]  tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': 'call_123', 'type': 'tool_call'}]
[2] ToolMessage   content='Sunny, 72°F'  tool_call_id=call_123
[3] AIMessage     content='The weather in San Francisco is sunny with a temperature of 72°F.'


---
## 🤖 Part 4: Messages in an Agent Run

An agent is a loop that appends messages to this same list. `create_agent` (introduced in
`03_react_with_create_agent.ipynb`, module 03) takes a state dict whose `messages` key holds
the conversation, and returns the state with **every** message the run produced — not just
the answer:

```python
{"messages": [{"role": "user", "content": "..."}]}
```

In [13]:
# ============================================================================
# AGENT DEFINITION: A single-tool weather agent
# ============================================================================
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get the weather for a city."""
    # Stubbed for the demo — swap in a real weather API call.
    return f"The weather in {city} is sunny."


agent = create_agent(
    model=llm,
    tools=[get_weather],                            # plain functions are valid tools
    system_prompt="You are a helpful assistant.",
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather like in New York?"}]}
)

print("🤖 Final answer:", response["messages"][-1].content)
print(f"📋 Messages in the returned state: {len(response['messages'])}")

🤖 Final answer: The weather in New York is sunny.
📋 Messages in the returned state: 4


### 4.1 ❓ Why Does One AI Message Look Empty?

Printed as plain text, the trail looks broken — the first AI turn is blank:

```
[human] What is the weather like in New York?
[ai]
[tool] The weather in New York is sunny.
[ai] The weather in New York is sunny.
```

That is not a bug: **that AI message *is* the tool call.**

| # | Message | What it carries |
|---|---------|-----------------|
| 1 | `HumanMessage` | your question |
| 2 | `AIMessage` | `content=""`, plus `tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_...'}]` |
| 3 | `ToolMessage` | `"The weather in New York is sunny."` — what the function returned |
| 4 | `AIMessage` | the final natural-language answer |

When the model decides to call a tool it emits *only* the call; there is no prose to say
alongside it, so `.content` is an empty string and the real payload lives in `.tool_calls`.
Print `.content` alone and you see a blank line.

> **Note**: some models (Anthropic, and OpenAI's reasoning models) *do* emit a short
> "Let me check the weather…" preamble next to the call. Never assume an `AIMessage` is
> either text or a tool call — it can be both, or neither.

In [14]:
# ============================================================================
# MESSAGE ANATOMY: Show content AND tool_calls side by side
# ============================================================================
for i, m in enumerate(response["messages"]):
    print(f"[{i}] {type(m).__name__:<12} content={m.content!r}")
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"     🔧 tool_call -> {tc['name']}({tc['args']})")

[0] HumanMessage content='What is the weather like in New York?'
[1] AIMessage    content=''
     🔧 tool_call -> get_weather({'city': 'New York'})
[2] ToolMessage  content='The weather in New York is sunny.'
[3] AIMessage    content='The weather in New York is sunny.'


### 4.2 ⌨️ The String Shorthand

Passing a bare string as `messages` is accepted shorthand — LangChain wraps it into a single
`HumanMessage` for you, exactly as `llm.invoke("...")` did in Part 2. Handy for quick
experiments, but prefer the explicit list form in real code, where you need system messages and
multi-turn history anyway.

In [15]:
# ============================================================================
# STRING SHORTHAND: {"messages": "..."} instead of a list of dicts
# ============================================================================
# Note the deliberate typo ("New Yourk") — the model still resolves the city,
# a nice illustration that tool ARGS are model-generated, not string-parsed.
response1 = agent.invoke({"messages": "What is the weather in New Yourk"})

for i, m in enumerate(response1["messages"]):
    calls = [f"{tc['name']}({tc['args']})" for tc in getattr(m, "tool_calls", None) or []]
    print(f"[{i}] {type(m).__name__:<12} {m.content!r} {calls or ''}")

[0] HumanMessage 'What is the weather in New Yourk' 
[1] AIMessage    '' ["get_weather({'city': 'New York'})"]
[2] ToolMessage  'The weather in New York is sunny.' 
[3] AIMessage    'The weather in New York is sunny.' 


---
## 📝 Summary

In this notebook, we learned:

### 1. Two Input Shapes
- **Text prompt**: a bare string, wrapped into a `HumanMessage` — fine for one-shot requests
- **Message list**: required as soon as you need a system prompt or conversation history

### 2. The Four Message Types
- **`SystemMessage`**: behaviour, role, and guidelines — the most leveraged message you write
- **`HumanMessage`**: user input, optionally tagged with `name` and `id`
- **`AIMessage`**: model output — text, tool calls, provider metadata
- **`ToolMessage`**: one tool's result, keyed back to its call

### 3. The List *Is* the State
- **No hidden memory**: the model sees exactly the list you pass, nothing more
- **You can forge history**: hand-built `AIMessage`s replay stored conversations seamlessly

### 4. Metadata Worth Reading
- **`usage_metadata`**: input/output/total tokens — log this to track cost
- **`tool_call_id`**: must match between the `AIMessage` tool call and its `ToolMessage`

### 5. Messages in an Agent Run
- **The agent returns the whole trail**: Human → AI (tool call) → Tool → AI (answer);
  `response["messages"][-1].content` is the answer
- **The "empty" AI message** holds the tool call in `.tool_calls`, not in `.content` — print
  both when debugging

### Next Steps
- **`03_structured_output.ipynb`** — force responses into a schema with Pydantic, `TypedDict`,
  and dataclasses via `with_structured_output` and `response_format`